In [ ]:
import os
import time
import pandas as pd
from sqlalchemy import create_engine
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display

# 1. Cargamos credenciales
load_dotenv(override=True)
pd.options.display.float_format = '{:.0f}'.format

# 2. IA y Base de Datos
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("LLM_API_KEY"))
engine = create_engine(f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

def ejecutar_consulta(sql):
    try:
        return pd.read_sql(sql, engine)
    except Exception as e:
        return f"Error de MySQL: {e}"

def consultar_biblioia(pregunta_usuario):
    prompt_sistema =  """
Sos un asistente experto en bases de datos MySQL para una biblioteca. 
Tu objetivo es traducir preguntas en lenguaje natural a consultas SQL precisas.

ESQUEMA DE LA BASE DE DATOS:
- Libro (Isbn, Titulo, Año, StockTotal, StockDisponible)
- Socio (Id, Dni, Nombre, Apellido, Mail, FechaAlta, IdSexo, IdEstadoSocio)
- Ejemplar (Numero,FechaAlta, IsbnLibro, IdEstadoEjemplar)
- Prestamo (Id,FechaPrestamo, FechaVencimiento, FechaDevolucion, IdSocio, IdEjemplar, IdEstadoPrestamo)
- Sancion (Id, FechaInicio, FechaFin, Motivo, IdTipoSancion, IdPrestamo, IdSocio)
- GeneroLibro(Id, Nombre, Descripcion)
- Autor(Id, Nombre, Apellido,IdSexo, IdNacionalidad)
- Nacionalidad(Id, Pais)
- Autor_Libro(IdAutor, IsbnLibro)
- GeneroLibro_Libro(IsbnLibro, IdGeneroLibro)
- TipoSancion(Id, Tipo)
- Sexo(Id, Tipo)
- EstadoSocio(Id, Tipo)
- EstadoEjemplar(Id, Tipo)
- EstadoPrestamo(Id, Tipo)

CONSTRAINTS Y ESTADOS:
- EstadoSocio: 1='Activo', 2='Suspendido', 3='Baja'
- EstadoEjemplar: 1='Disponible', 2='Prestado', 3='Baja'
- EstadoPrestamo: 1='Activo', 2='Devuelto', 3='Vencido'
- TipoSancion: 1='Leve', 2='Medio', 3='Grave'
- Sexo:  1='Femenino',2='Masculino', 3='Otro'

INSTRUCCIÓN CRÍTICA:
Respondé ÚNICA Y EXCLUSIVAMENTE con una consulta SQL válida para MySQL.
Devuelve solo la cadena de texto de la consulta, sin bloques de código Markdown ni explicaciones.
"""
    try:
        chat_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "system", "content": prompt_sistema}, {"role": "user", "content": pregunta_usuario}],
            temperature=0.1
        )
        return chat_completion.choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()
    except Exception as e:
        return f"Error: {e}"

def agente_responder(pregunta):
    print(f"🤖 Pensando SQL para: '{pregunta}'")
    sql = consultar_biblioia(pregunta)
    if sql.startswith("Error"):
        print("❌ Error IA.")
        return
    resultado = ejecutar_consulta(sql)
    if isinstance(resultado, pd.DataFrame):
        display(resultado.style.hide(axis="index"))
    else:
        print(f"❌ {resultado}")

In [5]:
print("🎯 INICIANDO BATERÍA DE 10 PRUEBAS DEL AGENTE IA 🎯\n")


consultas_prueba = [
    ("PRUEBA 1: Top 5", "¿Cuáles son los 5 libros más prestados este año?"),
    ("PRUEBA 2: Estado condicional", "¿Qué socios tienen préstamos vencidos en este momento?"),
    
   
    ("PRUEBA 3: Filtro específico", "¿Cuántos ejemplares disponibles hay del libro con ISBN 9780451524935?"), 
    
    ("PRUEBA 4: Doble condición", "¿Qué libros de ciencia ficción están disponibles para prestar?"),
    
   
    ("PRUEBA 5: Historial", "¿Cuál es el historial de préstamos del socio con DNI 43000444?"), 
    
    ("PRUEBA 6: HAVING / Conteo", "¿Qué autores tienen más de 3 libros en la biblioteca?"),
    ("PRUEBA 7: Filtro por fechas", "¿Cuántas sanciones se generaron en el último mes?"),
    ("PRUEBA 8: MAX / Subconsultas", "¿Quién es el socio con una mayor cantidad de sanciones?"),
    
   
    ("PRUEBA 9: Cruce complejo", "¿Cuál es el género más solicitado por el socio con DNI 43000444?"),
    
    ("PRUEBA 10: Matemáticas en SQL", "¿Porcentaje de préstamo por género en el último mes?") 
]

for titulo, pregunta in consultas_prueba:
    print(f"{'='*70}")
    print(f"🚀 {titulo}")
    print(f"👤 Pregunta: {pregunta}")
    print(f"{'='*70}")
    
    agente_responder(pregunta)

    time.sleep(3) 
    print("\n")
    
print("✅ PRUEBAS FINALIZADAS ✅")

🎯 INICIANDO BATERÍA DE 10 PRUEBAS DEL AGENTE IA 🎯

🚀 PRUEBA 1: Top 5
👤 Pregunta: ¿Cuáles son los 5 libros más prestados este año?
🤖 Pensando SQL para: '¿Cuáles son los 5 libros más prestados este año?'


Titulo,CantidadPrestamos
El camino de los reyes,4
Harry Potter y la piedra filosofal,4
Cien años de soledad,4
It,3
Palabras radiantes,2




🚀 PRUEBA 2: Estado condicional
👤 Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
🤖 Pensando SQL para: '¿Qué socios tienen préstamos vencidos en este momento?'


Id,Nombre,Apellido
3,Alex,Ríos
2,Juan,Pérez
26,Roberto,Sanz
27,Clara,Castro
28,Victor,Ortega
21,Antonio,Navarro
22,Laura,Herrero
23,Diego,Dominguez
24,Patricia,Gil
25,Sergio,Vazquez




🚀 PRUEBA 3: Filtro específico
👤 Pregunta: ¿Cuántos ejemplares disponibles hay del libro con ISBN 9780451524935?
🤖 Pensando SQL para: '¿Cuántos ejemplares disponibles hay del libro con ISBN 9780451524935?'


...



🚀 PRUEBA 9: Cruce complejo
👤 Pregunta: ¿Cuál es el género más solicitado por el socio con DNI 43000444?
🤖 Pensando SQL para: '¿Cuál es el género más solicitado por el socio con DNI 43000444?'


Nombre
Ficción




🚀 PRUEBA 10: Matemáticas en SQL
👤 Pregunta: ¿Porcentaje de préstamo por género en el último mes?
🤖 Pensando SQL para: '¿Porcentaje de préstamo por género en el último mes?'


Nombre,CantidadPrestamos,Porcentaje
Ficción,7,35.000000
Realismo Mágico,4,20.000000
Terror,2,10.000000
Policial,3,15.000000
Ciencia Ficción,4,20.000000




✅ PRUEBAS FINALIZADAS ✅
